## 3.2 Task: Continuous Price Prediction Modeling

In this task, we developed a regression model to predict the continuous price value of an e-commerce order. The goal is to estimate order price using the available transaction features. We used a Multiple Linear Regression approach trained with gradient descent through Scikit-learn's SGDRegressor.

This task includes:
- preparing the regression dataset
- building a preprocessing + regression pipeline
- experimenting with gradient descent parameters
- evaluating the model using MSE and MAE
- applying k-fold cross-validation
- analyzing underfitting and overfitting

In [ ]:
print("3.2 Task: Continuous Price Prediction Modeling")

In [ ]:
# Load data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/cleaned_data.csv")
df.head()

In [ ]:
# Define Features & Target (Regression)

# Regression target
y = df["price"]

# Input features
X = df.drop(columns=["price", "delivery_status", "customer_segment"], errors="ignore")

In [ ]:
# Identify Column Types

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

numerical_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

In [ ]:
# Build preprocessing pipeline
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# Split the data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


# Build baseline regression model
from sklearn.linear_model import SGDRegressor

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", SGDRegressor(
        loss="squared_error",
        learning_rate="constant",
        eta0=0.01,
        max_iter=1000,
        tol=1e-3,
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)

# Predict and evaluate baseline model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_train_pred = baseline_model.predict(X_train)
y_test_pred = baseline_model.predict(X_test)

train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Baseline Model Performance")
print("Train MSE:", train_mse)
print("Test MSE :", test_mse)
print("Train MAE:", train_mae)
print("Test MAE :", test_mae)
print("Train R² :", train_r2)
print("Test R²  :", test_r2)



In [ ]:
# Cross-validation

from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_mse_scores = -cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_squared_error"
)

cv_mae_scores = -cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

print("Cross-Validation Results")
print("CV MSE scores:", cv_mse_scores)
print("Average CV MSE:", cv_mse_scores.mean())
print("CV MAE scores:", cv_mae_scores)
print("Average CV MAE:", cv_mae_scores.mean())

In [ ]:
# Experiment with gradient descent settings

experiments = [
    {"eta0": 0.001, "max_iter": 500,  "learning_rate": "constant"},
    {"eta0": 0.001, "max_iter": 1000, "learning_rate": "constant"},
    {"eta0": 0.01,  "max_iter": 1000, "learning_rate": "constant"},
    {"eta0": 0.05,  "max_iter": 1000, "learning_rate": "constant"},
    {"eta0": 0.01,  "max_iter": 2000, "learning_rate": "constant"},
    {"eta0": 0.01,  "max_iter": 1000, "learning_rate": "optimal"},
]

results = []

for exp in experiments:
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", SGDRegressor(
            loss="squared_error",
            eta0=exp["eta0"],
            max_iter=exp["max_iter"],
            learning_rate=exp["learning_rate"],
            tol=1e-3,
            random_state=42
        ))
    ])
    
    pipe.fit(X_train, y_train)
    
    train_pred = pipe.predict(X_train)
    test_pred = pipe.predict(X_test)
    
    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)
    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    
    cv_mse = -cross_val_score(
        pipe, X_train, y_train,
        cv=cv, scoring="neg_mean_squared_error"
    ).mean()
    
    results.append({
        "learning_rate_type": exp["learning_rate"],
        "eta0": exp["eta0"],
        "epochs_max_iter": exp["max_iter"],
        "train_mse": train_mse,
        "test_mse": test_mse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "cv_mse": cv_mse
    })

    
results_df = pd.DataFrame(results).sort_values(by="test_mse")
results_df

### Note on batch size
Scikit-learn's `SGDRegressor` performs stochastic gradient descent internally and does not provide a direct `batch_size` parameter like TensorFlow or PyTorch. Therefore, in this notebook we experimentally varied the learning rate and the number of training iterations (`max_iter`) to study gradient descent behavior. In the conceptual discussion, batch size is still explained because it is part of the theoretical requirement of the task.

In [ ]:
# Select best model

best_row = results_df.iloc[0]
best_row

best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", SGDRegressor(
        loss="squared_error",
        eta0=best_row["eta0"],
        max_iter=int(best_row["epochs_max_iter"]),
        learning_rate=best_row["learning_rate_type"],
        tol=1e-3,
        random_state=42
    ))
])

best_model.fit(X_train, y_train)
best_test_pred = best_model.predict(X_test)

best_test_mse = mean_squared_error(y_test, best_test_pred)
best_test_mae = mean_absolute_error(y_test, best_test_pred)

print("Best Model Test MSE:", best_test_mse)
print("Best Model Test MAE:", best_test_mae)

In [ ]:
# Visualization: actual vs predicted

plt.figure(figsize=(8, 4))
plt.plot(y_test.values[:100], label="Actual", marker="o")
plt.plot(best_test_pred[:100], label="Predicted", marker="x")
plt.title("Actual vs Predicted Prices (First 100 Test Samples)")
plt.xlabel("Sample Index")
plt.ylabel("Price")
plt.legend()
plt.show()

## Effect of Learning Rate, Epochs, and Batch Size

### Learning Rate
The learning rate controls how large each update step is during gradient descent.  
- A very small learning rate makes learning slow but stable.
- A very large learning rate may cause unstable updates or overshooting.
- In our experiments, moderate values gave better test performance than extremely small or large values.

### Epochs
In Scikit-learn's SGDRegressor, `max_iter` represents the maximum number of passes over the training data.
- Too few iterations may lead to underfitting because the model has not learned enough.
- Too many iterations may increase training time and may sometimes lead to overfitting if the model becomes too adapted to the training data.

### Batch Size
Batch size refers to the number of samples processed before updating the model weights.
- Small batch sizes produce noisier but faster updates.
- Large batch sizes produce smoother but slower updates.
- Scikit-learn's SGDRegressor does not directly expose a batch size parameter, so our implementation focuses mainly on learning rate and iteration tuning while discussing batch size conceptually.

## MSE and MAE Interpretation

Mean Squared Error (MSE) measures the average squared difference between actual and predicted values. Because the errors are squared, large errors are penalized more heavily.

Mean Absolute Error (MAE) measures the average absolute difference between actual and predicted values. It is easier to interpret because it shows the average error in the same unit as the target price.

In the AuraCart business context:
- A high MSE means some orders are being predicted very badly, which can harm revenue forecasting.
- A high MAE means the average pricing prediction error is large, which reduces trust in the system.

Large pricing errors may lead to poor financial planning, inaccurate forecasting, and weaker operational decisions.

## Underfitting and Overfitting Analysis

To determine whether the model underfits or overfits, we compared training error, testing error, and cross-validation error.

- If both training and testing errors are high, the model is likely underfitting.
- If training error is low but testing error is much higher, the model is likely overfitting.
- If training, testing, and cross-validation errors are reasonably close, the model is generalizing well.

In our results, the comparison between training MSE, testing MSE, and average cross-validation MSE helps us decide whether the model is stable enough for price prediction.

In [ ]:
# Save final regression pipeline

import joblib

joblib.dump(best_model, "../artifacts/regression_model.joblib")
print("Best regression model saved successfully.")